In [ ]:
# OpenBLAS starts one thread per core even for tiny BLAS-1 calls, and the Krylov
# solvers below issue thousands of those per iteration. On a 24-core machine the
# oversubscription cost 4.7x. This has to be set *before* numpy is imported, so if
# numpy is already loaded, restart the kernel.
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import splu, cg, bicgstab, LinearOperator
%matplotlib qt5


In [ ]:
# Inputs
# ------

# Immersed body (circular cylinder) and doubly-periodic domain
R  = 1
Lx = 10
Ly = 10
nx = 81
ny = 81

# Initial timestep (the time derivative acts as the SIMPLEC under-relaxation)
dt = 0.1

# Kinematic viscosity. VAR[IP] is the reduced pressure p/rho, so the density
# never appears explicitly in the equations below.
nu = 1.0

# Constant-flow-rate forcing: the uniform body force in x is adjusted every
# iteration so that the bulk velocity relaxes onto this target. Free slip
# generates no friction drag, so with a *fixed* force the flow would keep
# accelerating and no steady state would exist.
uBulkTarget = 0.5

# Linear solvers. Both give the same answer; only the cost differs.
#   'direct' - SuperLU factorisation of every system, every iteration.
#   'krylov' - BiCGStab on the momentum, preconditioned by a factorisation reused
#              across iterations; CG on the pressure, preconditioned by a
#              constant-coefficient FFT Poisson solve.
# Measured with no-slip (s per iteration, 10 iterations, one thread):
#     16k cells   direct 0.15   krylov 0.18
#     66k cells   direct 1.06   krylov 1.08      <- crossover
#    262k cells   direct 12.8   krylov 7.0
# The FFT preconditioner is much cheaper when nx and ny have small prime factors
# (161 = 7*23 costs about twice as much per FFT as 160).
# Cut-cell (partial-face-area) route. The boundary condition is carried entirely by
# the geometric closure identity, so there is no ghost value, no blanking of cut
# cells and no cross term between the velocity components -- and none of the
# stabilisation the ghost route needs. See doc/freeslip_ibm.pdf, Part II.
#
# Because the cross term is what distinguished the two momentum routes, the coupled
# and segregated variants here give bit-identical results.

linearSolver = 'direct'
pTol         = 1e-12       # CG relative tolerance for the pressure equation

# Apply the free-slip ghost to the CONVECTIVE stencil as well as the viscous one.
#
# LEAVE THIS FALSE. It is required for consistency on an oblique wall -- with it,
# uniform flow along a 45-degree free-slip wall is exact (3.7e-09); without it the
# error is O(1) (7.4e-01) -- but every treatment tried so far is unstable on a
# curved body: lagged, lagged with a diagonal repair, with h/delta clipped, and
# fully implicit in the coupled matrix below. Diverges at nx = 41 and 61 in all of
# them, at every timestep tried (10, 1, 0.1), and step rejection does not help.
#
# Cause: the ghost adds  0.5*f*(h/delta)*(n n^T)  to the 2x2 block, where f is the
# face flux. n n^T is rank-1 positive semi-definite, so the *viscous* copy (which
# carries +nu instead of f) is well posed and is exactly what makes delta -> 0 safe
# in Luchini et al. But f changes sign, so the convective copy is indefinite, and
# making an indefinite block implicit does not make it well posed.
#
# Likely cure, not yet tried: upwind the convective ghost, so the ghost value only
# enters on inflow faces and f contributes with a definite sign.
ghostConvection = True

In [ ]:
# Grid, fields and sparsity pattern
# ---------------------------------
IU, IV, IP = 0, 1, 2                 # component indices into VAR

dx, dy = Lx/nx, Ly/ny
V      = dx*dy                       # cell volume
nCells = nx*ny

xp = (np.arange(nx)+0.5)*dx          # cell centres
yp = (np.arange(ny)+0.5)*dy
xf = np.arange(nx+1)*dx              # face centres
yf = np.arange(ny+1)*dy
XP, YP = np.meshgrid(xp, yp)
XF, YF = np.meshgrid(xf, yf)

VAR = np.zeros((3, ny, nx))          # u, v, p

# The 5-point stencil is held as five (ny,nx) coefficient arrays in the order
# P, E, W, N, S -- not as an (nCells,nCells) matrix. Its sparsity pattern never
# changes, so the CSR structure and the permutation from stencil order into CSR
# data order are built once, here. Assembling a matrix is then an O(nnz)
# fancy-index rather than an O(nCells^2) scan over a 99.92%-empty dense array.
ID  = np.arange(nCells).reshape(ny, nx)
ROW = np.stack([ID]*5, -1).ravel()
COL = np.stack([ID, np.roll(ID, -1, 1), np.roll(ID, 1, 1),
                    np.roll(ID, -1, 0), np.roll(ID, 1, 0)], -1).ravel()
assert len(set(zip(ROW.tolist(), COL.tolist()))) == ROW.size, \
    'stencil aliases itself: nx and ny must both be at least 3'

_pattern    = csr_matrix((np.arange(1.0, ROW.size+1), (ROW, COL)),
                         shape=(nCells, nCells))
CSR_PERM    = (_pattern.data-1).astype(np.int64)
CSR_INDICES = _pattern.indices
CSR_INDPTR  = _pattern.indptr


# Coupled (u,v) system. The free-slip cross term couples the two components at the
# *same* cell only, so the block matrix
#
#     [ Auu   diag(cU) ] [u]   [bu]
#     [ diag(cV)   Avv ] [v] = [bv]
#
# has just one extra entry per row: 6 nnz instead of 5. Solving it directly removes
# the lag on the cross term altogether, which is what makes the curved-body case
# stable (lagging it diverges at nx = 41 and 61 whatever the timestep).
IDX  = np.arange(nCells)
ROW2 = np.concatenate([ROW, ROW+nCells, IDX, IDX+nCells])
COL2 = np.concatenate([COL, COL+nCells, IDX+nCells, IDX])


def toCSR2(aU, aV, cU, cV):
    """Assemble the coupled 2N x 2N momentum matrix."""
    dat = np.concatenate([aU.reshape(5, -1).T.ravel(), aV.reshape(5, -1).T.ravel(),
                          cU.ravel(), cV.ravel()])
    return csr_matrix((dat, (ROW2, COL2)), shape=(2*nCells, 2*nCells))


def zeroStencil():
    """An empty set of stencil coefficients, indexed [P|E|W|N|S, iy, ix]."""
    return np.zeros((5, ny, nx))


def toCSR(a):
    """Assemble stencil coefficients into a sparse matrix."""
    return csr_matrix((a.reshape(5, -1).T.ravel()[CSR_PERM], CSR_INDICES, CSR_INDPTR),
                      shape=(nCells, nCells))


def offDiagDot(a, phi):
    """(A - diag(aP)) @ phi, as four neighbour shifts. No matrix is formed."""
    return (a[1]*np.roll(phi, -1, 1) + a[2]*np.roll(phi, 1, 1)
          + a[3]*np.roll(phi, -1, 0) + a[4]*np.roll(phi, 1, 0))


def wrapX(f):
    """Cell-array -> face-array in x: append the periodic image of the first face."""
    return np.concatenate([f, f[:, :1]], axis=1)


def wrapY(f):
    return np.concatenate([f, f[:1, :]], axis=0)

## Free-slip immersed boundary — partial face areas (cut cells)

Collocated finite volumes, implicit Euler, SIMPLEC as in OpenFOAM's `simpleFoam`, free slip
on the body. This is the **cut-cell** route. The companion pair `SIMPLEC.ipynb` /
`SIMPLEC_Coupled.ipynb` implement the **ghost** route; both are documented in
`doc/freeslip_ibm.pdf`.

Every face carries its fluid area fraction $\theta\in[0,1]$ and every cell its fluid volume
fraction $\alpha$. These replace the boolean open/closed mask: the body is opaque again, but
a face the boundary cuts stays *partly* open, so tangential flow is not blocked.

### The identity that does all the work

For the fluid polygon of a cut cell,
$$\sum_f \theta_f\,\mathbf{n}_f A_f + \mathbf{n}_{\rm wall}A_{\rm wall}=0 ,$$
to machine precision, because $\theta$ and the wall segment come from the same level set.
That single closure relation carries the entire boundary condition:

| term | wall contribution | consequence |
|---|---|---|
| viscous | zero (free slip $\Rightarrow$ no tangential stress) | $\theta$-weighted face sum; annihilates a constant field |
| convective | zero (no penetration $\Rightarrow$ no wall flux) | $\theta$-weighted face sum |
| pressure | $p_{\rm wall}\mathbf{n}_{\rm wall}A_{\rm wall}$ — the form drag | closure turns the Gauss gradient into $\sum_f\theta_f(p_f-p_P)\mathbf{n}_fA_f$, zero for uniform $p$ |
| continuity | zero | $\sum_f\theta_f\phi_f=0$; with closure this forces $\mathbf{u}\!\cdot\!\mathbf{n}_{\rm wall}=0$ |

**No penetration is never imposed anywhere — it emerges** from continuity plus closure. So
this route has **no ghost value, no blanking of cut cells, no cross term** between the
velocity components, and needs none of the S1/S3 stabilisation the ghost route requires. The
cross term vanishing is also why the coupled and segregated variants here produce
*bit-identical* results: with no cross coupling they solve the same block-diagonal system.

### Verification

45° free-slip channel, exact solution uniform flow with $g_x=0$:

| | ghost route | **cut cells** |
|---|---|---|
| drag (exact 0) | $-4.9\times10^{-9}$ | $\mathbf{0.000000}$ |
| mean $|\nabla\!\cdot\!\phi|$, fluid | $4\times10^{-6}$ | $\mathbf{2\times10^{-18}}$ |
| mean $|\nabla\!\cdot\!\phi|$, first solid layer | $4.8\times10^{-3}$ | $\mathbf{0}$ |

Cylinder drag, all converged to $10^{-9}$, symmetry held to $10^{-14}$, continuity machine-zero:

| $n_x$ | 41 | 61 | 81 | 121 |
|---|---|---|---|---|
| $\nu=1.0$ ($Re_R=0.5$) | 2.829667 | 2.907463 | 2.938772 | 2.977790 |
| $\nu=0.5$ ($Re_R=1.0$) | 1.449213 | 1.487457 | 1.503127 | 1.523026 |

### The one cost

Cut cells with $\alpha\to0$ have vanishing volume against $O(h)$ faces — the classic
small-cell problem. `ALPHA_MIN` absorbs slivers into the body, which is the only place this
route gives ground: it moves the boundary by $O(\alpha_{\min}h)$, and it must zero the faces
of an absorbed cell too, or geometric closure — and with it the exactness above — is lost.
Cell merging would remove even that. The drag sequences are monotone but decreasing in
increment only slowly, consistent with the first-order boundary placement.

In [ ]:
# Finite-volume operators on the cut mesh
# ---------------------------------------
# Every face carries its fluid area fraction theta, every cell its fluid volume
# fraction alpha. The fluid part of a cut cell is a polygon whose boundary is the
# partially-open axis faces plus the wall segment, and geometric closure
#
#     sum_f theta_f n_f A_f + n_wall A_wall = 0                              (*)
#
# holds to machine precision because theta and the wall segment come from the same
# level set. That single identity carries the whole free-slip boundary condition:
#
#   * viscous   -- free slip means zero tangential stress on the wall segment, and
#                  at first order no normal viscous stress either, so the wall
#                  contributes nothing and the operator is just the theta-weighted
#                  face sum. It annihilates a constant field, as it must.
#   * convective-- no penetration means no flux through the wall segment, so again
#                  only the theta-weighted faces contribute.
#   * pressure  -- the wall DOES carry a pressure force (this is the form drag).
#                  With p_wall = p_P to first order, (*) turns the Gauss gradient
#                  into  sum_f theta_f (p_f - p_P) n_f A_f, which vanishes for a
#                  uniform pressure, as it must.
#   * continuity-- sum_f theta_f phi_f = 0. Combined with (*), a uniform velocity
#                  in a cut cell satisfies this only if u.n_wall = 0: no penetration
#                  is not imposed anywhere, it EMERGES from continuity plus closure.
#
# So there is no ghost value, no blanking of cut cells, and no cross term coupling
# the velocity components. All of that machinery existed only to compensate for
# the boolean open/closed mask.


def fvm_ddt_LHS(a, dt):
    a[0] += alphaV/dt


def fvm_ddt_RHS(rhs, phi, dt):
    rhs += alphaV*phi/dt


def fvm_lap(a, GammaX, GammaY=None):
    """theta-weighted implicit laplacian; no wall term (zero tangential stress)."""
    if GammaY is None:
        GammaY = GammaX
    gE = 0.5*(GammaX+np.roll(GammaX, -1, 1))*(dy/dx)*thX[:, 1:]
    gW = 0.5*(GammaX+np.roll(GammaX,  1, 1))*(dy/dx)*thX[:, :-1]
    gN = 0.5*(GammaY+np.roll(GammaY, -1, 0))*(dx/dy)*thY[1:, :]
    gS = 0.5*(GammaY+np.roll(GammaY,  1, 0))*(dx/dy)*thY[:-1, :]
    a[0] -= gE+gW+gN+gS
    a[1] += gE
    a[2] += gW
    a[3] += gN
    a[4] += gS


def fvm_divConvFlux(a, phiX, phiY):
    """theta-weighted central div(phi u); phi already carries the wet face area."""
    fE, fW = phiX[:, 1:], phiX[:, :-1]
    fN, fS = phiY[1:, :], phiY[:-1, :]
    a[0] += 0.5*(fE-fW+fN-fS)
    a[1] += 0.5*fE
    a[2] -= 0.5*fW
    a[3] += 0.5*fN
    a[4] -= 0.5*fS


def fvc_grd(phi):
    """
    Volume-integrated Gauss gradient including the wall pressure force, written via
    the closure identity as sum_f theta_f (p_f - p_P) n_f A_f. Exactly zero for a
    uniform pressure, so no spurious force is generated on a cut cell.
    """
    dE = 0.5*(np.roll(phi, -1, 1)-phi)*thX[:, 1:]
    dW = 0.5*(np.roll(phi,  1, 1)-phi)*thX[:, :-1]
    dN = 0.5*(np.roll(phi, -1, 0)-phi)*thY[1:, :]
    dS = 0.5*(np.roll(phi,  1, 0)-phi)*thY[:-1, :]
    return np.stack([dy*(dE-dW), dx*(dN-dS)])


def fvc_grdFlux(phi, GammaX, GammaY=None):
    """Face fluxes of fvm_lap: theta-weighted, and matching it coefficient for coefficient."""
    if GammaY is None:
        GammaY = GammaX
    fx = (phi-np.roll(phi, 1, 1))/dx*0.5*(GammaX+np.roll(GammaX, 1, 1))*dy
    fy = (phi-np.roll(phi, 1, 0))/dy*0.5*(GammaY+np.roll(GammaY, 1, 0))*dx
    return wrapX(fx)*thX, wrapY(fy)*thY


def fvc_divFlux(phiX, phiY):
    return (phiX[:, 1:]-phiX[:, :-1]) + (phiY[1:, :]-phiY[:-1, :])


def interpolatePhi(Ux, Uy):
    """Face volume fluxes through the wet part of each face."""
    return (0.5*(wrapX(np.roll(Ux, 1, 1))+wrapX(Ux))*dy*thX,
            0.5*(wrapY(np.roll(Uy, 1, 0))+wrapY(Uy))*dx*thY)

In [ ]:
# Immersed boundary: cut cells
# ----------------------------
def levelSet(x, y):
    """Negative inside the body, zero on the surface."""
    return (x-Lx/2)**2 + (y-Ly/2)**2 - R**2


ALPHA_MIN = 0.02   # cells with less fluid than this are absorbed into the body,
                   # which moves the boundary by O(ALPHA_MIN*h) and keeps the
                   # small-cell problem out of the linear systems

# Cut-cell geometry: face fluid fractions and cell fluid fractions
# ---------------------------------------------------------------
# theta_f in [0,1] is the fraction of face f that lies in the fluid; alpha is the
# same for the cell volume. They replace the boolean open/closed mask: the body is
# opaque again (no flux crosses the solid), but a face that the boundary cuts stays
# partly open, so tangential flow is not blocked.
#
# The property that makes this work is geometric closure. For the fluid polygon of
# a cut cell,
#
#     sum_f theta_f n_f A_f  +  n_wall A_wall = 0 ,
#
# so for a uniform field parallel to the wall (u.n_wall = 0) the discrete divergence
# sum_f theta_f (u.n_f) A_f = -(u.n_wall) A_wall vanishes *identically*. No ghost is
# needed for that to hold, and it holds however oblique the wall is.

def segmentFluidFraction(pA, pB, nBisect=40):
    """
    Fluid fraction of a straight segment whose level-set values at the two ends are
    pA and pB, found by bisection on the parametrised segment. Assumes at most one
    crossing per segment, which holds once the body is resolved.
    """
    fa, fb = pA(0.0), pB(1.0)
    lo = np.zeros_like(fa)
    hi = np.ones_like(fa)
    for _ in range(nBisect):
        mid = 0.5*(lo+hi)
        fm = pA(mid)
        sameAsA = (fm > 0.0) == (fa > 0.0)
        lo = np.where(sameAsA, mid, lo)
        hi = np.where(sameAsA, hi, mid)
    tCross = 0.5*(lo+hi)
    frac = np.where(fa > 0.0, tCross, 1.0-tCross)     # fluid share
    both = (fa > 0.0) == (fb > 0.0)
    return np.where(both, np.where(fa > 0.0, 1.0, 0.0), np.clip(frac, 0.0, 1.0))


def faceFractions():
    """theta on the x-faces (ny, nx+1) and the y-faces (ny+1, nx)."""
    XF1 = np.tile(xf, (ny, 1))
    Y0 = np.tile(yf[:-1][:, None], (1, nx+1))
    Y1 = np.tile(yf[1:][:, None], (1, nx+1))
    thX = segmentFluidFraction(lambda t: levelSet(XF1, Y0+t*(Y1-Y0)),
                               lambda t: levelSet(XF1, Y0+t*(Y1-Y0)))
    YF1 = np.tile(yf[:, None], (1, nx))
    X0 = np.tile(xp*0 + xf[:-1], (ny+1, 1))
    X1 = np.tile(xp*0 + xf[1:], (ny+1, 1))
    thY = segmentFluidFraction(lambda t: levelSet(X0+t*(X1-X0), YF1),
                               lambda t: levelSet(X0+t*(X1-X0), YF1))
    # x=0 and x=Lx are the same periodic face; likewise y. Compute once, copy, so
    # that conservation cannot be broken by an inconsistent seam.
    thX[:, -1] = thX[:, 0]
    thY[-1, :] = thY[0, :]
    return thX, thY


def cellFraction(nSub=32):
    """alpha, the fluid fraction of each cell, by sub-sampling the level set."""
    s = (np.arange(nSub)+0.5)/nSub
    acc = np.zeros((ny, nx))
    for a in s:
        for b in s:
            acc += (levelSet(XF[:-1, :-1]+a*dx, YF[:-1, :-1]+b*dy) > 0.0)
    return acc/(nSub*nSub)


thX, thY = faceFractions()
alpha = cellFraction()

# Absorb slivers: a deactivated cell must also have its faces closed, or the
# discrete divergence would see a flux into a cell that carries no equation.
tiny = alpha < ALPHA_MIN
alpha = np.where(tiny, 0.0, alpha)
thX[:, :-1] = np.where(tiny, 0.0, thX[:, :-1]); thX[:, 1:] = np.where(tiny, 0.0, thX[:, 1:])
thY[:-1, :] = np.where(tiny, 0.0, thY[:-1, :]); thY[1:, :] = np.where(tiny, 0.0, thY[1:, :])

fluid  = alpha > 0.0
solid  = ~fluid
alphaV = alpha*V
nearBody = fluid & (np.roll(solid, 1, 1) | np.roll(solid, -1, 1) |
                    np.roll(solid, 1, 0) | np.roll(solid, -1, 0))
IBM_SOLID_WEIGHT = 1e12


def ibm_blankSolid(a):
    """Cells with no fluid carry no equation."""
    a[0] += np.where(solid, IBM_SOLID_WEIGHT, 0.0)

In [ ]:
# Linear solvers
# --------------
# Both classes below only ever *precondition*. The matrices that are actually
# solved -- and therefore pEqn.flux() and the whole SIMPLEC splitting -- are
# untouched, so the algorithm stays exactly the OpenFOAM one.

class ReusedLU:
    """
    A SuperLU factorisation kept as a preconditioner across SIMPLE iterations.

    The momentum matrix changes a little every iteration (through the convective
    flux and dt), so the factorisation goes stale -- but a stale LU is still an
    excellent preconditioner, and the Krylov solve remains exact. Refactorise only
    when the iteration count says the old one has decayed. Because the LU is never
    used as the solver, the answer cannot depend on how stale it is.
    """

    def __init__(self, krylov, maxIts=20, rtol=1e-12):
        self.krylov, self.maxIts, self.rtol = krylov, maxIts, rtol
        self.lu, self.nFactor, self.nSolve, self.nIts = None, 0, 0, 0

    def solve(self, A, b, x0=None):
        if self.lu is None:
            self.lu = splu(A.tocsc()); self.nFactor += 1
        for attempt in range(2):
            its = [0]
            x, info = self.krylov(
                A, b, x0=x0, rtol=self.rtol, maxiter=500,
                M=LinearOperator(A.shape, matvec=self.lu.solve),
                callback=lambda xk: its.__setitem__(0, its[0]+1))
            if info == 0 and its[0] <= self.maxIts:
                break
            if attempt == 0:                                   # stale: refresh, retry
                self.lu = splu(A.tocsc()); self.nFactor += 1
        assert info == 0, f'momentum {self.krylov.__name__} failed, info={info}'
        self.nSolve += 1
        self.nIts += its[0]
        return x


class FftPoissonPC:
    """
    Preconditioner for the pressure equation: the *constant*-coefficient periodic
    Poisson operator with the same 5-point stencil, inverted exactly by FFT in
    O(N log N), wrapped in a symmetric Jacobi scaling.

    The Jacobi scaling is what makes this work at all: without it the pressure
    matrix has condition number ~1e17, because rAtU is ~1e-12 inside the solid
    (the IBM penalty) against ~1e2 in the fluid. That is a pure row/column scaling
    artefact -- symmetric Jacobi scaling alone brings it to ~5e4 -- and the FFT
    then removes the smooth error that Jacobi leaves behind.
    """

    def __init__(self):
        tx = 2*np.pi*np.fft.fftfreq(nx)
        ty = 2*np.pi*np.fft.fftfreq(ny)
        TX, TY = np.meshgrid(tx, ty)
        self.symbol = -2*((dy/dx)*(np.cos(TX)-1) + (dx/dy)*(np.cos(TY)-1))
        self.symbol[0, 0] = self.symbol[self.symbol > 0].min()   # keep it SPD

    def update(self, A, GammaBar):
        self.lam = GammaBar*self.symbol
        self.s   = 1.0/np.sqrt(np.abs(A.diagonal()))

    def matvec(self, r):
        z = self.s*r
        z = np.real(np.fft.ifft2(np.fft.fft2(z.reshape(ny, nx))/self.lam)).ravel()
        return self.s*z


def solvePressure(A, b, x0, pc):
    """CG on the SPD system (-A)p = (-b); fvm_lap has a negative diagonal."""
    its = [0]
    p, info = cg(-A, -b, x0=x0, rtol=pTol, maxiter=2000,
                 M=LinearOperator(A.shape, matvec=pc.matvec),
                 callback=lambda xk: its.__setitem__(0, its[0]+1))
    assert info == 0, f'pressure CG failed, info={info}'
    return p, its[0]

In [ ]:
# Run controls
# ------------
startNew    = True
readRestart = False
consistent  = True          # True: SIMPLEC, False: SIMPLE

nIterations = 400
cflMax      = 100.0         # Courant cap on the pseudo-transient relaxation
foMax       = 65.0          # diffusive (Fourier) cap, nu*dt/h^2
dtMax       = 10.0          # hard cap
dtRelax     = 1.0           # backoff factor, driven by the residual (see the loop)
residOld    = np.inf
dtCflPrev   = 1.0

# Initial condition
if readRestart and not startNew:
    try:
        VAR      = np.fromfile('var.fld').reshape(VAR.shape)
        phiHbyAx = np.fromfile('phiHbyAx.fld').reshape((ny, nx+1))
        phiHbyAy = np.fromfile('phiHbyAy.fld').reshape((ny+1, nx))
    except (FileNotFoundError, ValueError):
        print('No restart field found, starting from rest.')
        startNew = True
if startNew:
    VAR[...] = 0.0
    phiHbyAx, phiHbyAy = interpolatePhi(VAR[IU], VAR[IV])

# Static part of the system. One laplacian serves both components: free slip means
# the wall segment carries no viscous stress, so the operator is just the
# theta-weighted face sum with no wall term at all.
lapl = zeroStencil()
fvm_lap(lapl, nu*np.ones((ny, nx)))


# Pressure reference, pinned in a fluid cell. With every cut face closed the fluid
# region is a pure-Neumann island, so it needs exactly one reference -- and would
# need one per region if the body ever split the fluid in two.
jRef, iRef = divmod(int(np.flatnonzero(fluid.ravel())[0]), nx)

Afluid     = alphaV.sum()
unitSource = np.stack([alphaV, np.zeros((ny, nx))])   # source for gx = 1
uSolver    = ReusedLU(bicgstab)              # one coupled (u,v) matrix
pPC        = FftPoissonPC()
t          = 0.0


def bulkVelocity(u):
    return np.sum(u*alphaV)/Afluid


# Timeloop
for it in range(nIterations):

    VARold  = VAR.copy()
    phiOldX, phiOldY = phiHbyAx.copy(), phiHbyAy.copy()

    # Assemble the operators. None of this depends on the driving force.
    # -----------------------------------------------------------------
    aMom = []
    for iC in range(2):
        a = zeroStencil()
        fvm_ddt_LHS(a, dt)                            # d/dt
        a -= lapl                                 # -laplacian(nu, u), IBM-corrected

        aMom.append(a)
    for a in aMom:
        fvm_divConvFlux(a, phiHbyAx, phiHbyAy)
    for a in aMom:
        ibm_blankSolid(a)                             # u = 0 inside the body

    # Split as  aP*u = H(u) - V*grad(p),  H(u) = source - offDiag*u
    aP     = np.stack([a[0] for a in aMom])                # OpenFOAM A(), times V
    rAU    = 1.0/aP
    sumOff = np.stack([a[1:].sum(axis=0) for a in aMom])   # OpenFOAM -H1(), times V
    # Floor the SIMPLEC mobilities at the pure pseudo-transient value alpha*V/dt,
    # which is what a cell with this much fluid would have with no other coupling.
    # On this route the diagonal is already positive by construction; the floor only
    # guards the sliver cells left by ALPHA_MIN, where alpha*V is very small.
    aP     = np.maximum(aP, alphaV/dt)
    rAU    = 1.0/aP
    rAtU   = 1.0/np.maximum(aP+sumOff, alphaV/dt) if consistent else rAU
    rAtUV  = rAtU*V

    # Pressure matrix. rAtU differs between u and v under free slip, so x-faces use
    # the u mobility and y-faces the v one. (OpenFOAM instead averages the
    # components in A(); see addCmptAvBoundaryDiag.)
    pStencil = zeroStencil()
    fvm_lap(pStencil, rAtUV[IU], rAtUV[IV])
    # Every face of a solid cell is closed, so its row came out empty: give it an
    # identity row, with the sign of the negative-definite laplacian.
    pStencil[0][solid] = -1.0
    pStencil[0][solid] = -1.0
    pStencil[0, jRef, iRef] *= 2.0                    # OpenFOAM fvMatrix::setReference:
    Ap = toCSR(pStencil)                              # symmetric, unlike replacing the row

    # Factorise once; both passes below reuse it.
    crossU = crossV = np.zeros((ny, nx))
    Amom = toCSR2(aMom[0], aMom[1], crossU, crossV)
    if linearSolver == 'direct':
        luMom, luP = splu(Amom.tocsc()), splu(Ap.tocsc())
        solveMom = luMom.solve
        solvePres = lambda b: (luP.solve(b), 0)
    else:
        pPC.update(Ap, rAtUV[:, fluid].mean())
        solveMom = lambda b: uSolver.solve(Amom, b)
        solvePres = lambda b: solvePressure(Ap, b, None, pPC)

    def advance(sourceField, pOld):
        """
        One predictor-projection pass with the operators above. The whole pass is
        affine in its source, which is what the flow-rate constraint exploits.
        """
        RHS   = sourceField.reshape(2, -1)
        gradP = fvc_grd(pOld)
        w = solveMom(np.concatenate([RHS[0]-gradP[0].ravel(),
                                     RHS[1]-gradP[1].ravel()]))
        U = np.stack([w[:nCells].reshape(ny, nx), w[nCells:].reshape(ny, nx)])
        # H(u) now also carries the cross coupling, exactly as OpenFOAM's H() does
        HbyA = np.stack([(sourceField[IU] - offDiagDot(aMom[0], U[IU]) - crossU*U[IV])*rAU[IU],
                         (sourceField[IV] - offDiagDot(aMom[1], U[IV]) - crossV*U[IU])*rAU[IV]])
        pX, pY = interpolatePhi(HbyA[IU], HbyA[IV])
        if consistent:
            fX, fY = fvc_grdFlux(pOld, (rAtU[IU]-rAU[IU])*V, (rAtU[IV]-rAU[IV])*V)
            pX, pY = pX+fX, pY+fY
            HbyA = HbyA - (rAU-rAtU)*gradP
        div0 = fvc_divFlux(pX, pY)
        rhsP = div0.copy()
        rhsP[solid] = 0.0
        rhsP[solid] = 0.0
        pNew, its = solvePres(rhsP.ravel())
        pNew = pNew.reshape(ny, nx)
        fX, fY = fvc_grdFlux(pNew, rAtUV[IU], rAtUV[IV])
        return (HbyA - rAtU*fvc_grd(pNew), pNew, pX-fX, pY-fY, div0, its)

    # Constant flow rate, by linear superposition
    # ------------------------------------------
    # Only the source term depends on the driving force gx, and the pass above is
    # affine in it. So one pass with gx = 0 and one with gx = 1 (and everything
    # else zeroed) give the exact gx that lands the bulk velocity on its target --
    # no gain to tune. Free slip leaves the bulk mode almost undamped, so a
    # proportional controller on gx is not a safe alternative here.
    rhs = np.zeros((2, ny, nx))
    for iC in range(2):
        fvm_ddt_RHS(rhs[iC], VAR[iC], dt)
    # Free-slip cross term: the wall value of one component involves the other.
    # Lagged one outer iteration, which keeps the two systems uncoupled and is
    # exact once the outer iteration has converged.
    #
    # Lagging alone is unstable: the cross weight is d(-1)(h/delta) nX nY against a
    # diagonal correction d(-1)(h/delta) nX**2, so wherever |nY| > |nX| on a cut
    # x-arm the explicit term dominates the implicit one and the outer iteration
    # stalls (measured: residual stuck near 2e-4 by nx=121). So |cross| is added to
    # the diagonal and taken off again on the right with the lagged value of the
    # *same* component: the fixed point is untouched, but the iteration matrix
    # regains its diagonal dominance.

    base = advance(rhs,        VAR[IP])
    resp = advance(unitSource, np.zeros((ny, nx)))
    gx   = (uBulkTarget-bulkVelocity(base[0][IU]))/bulkVelocity(resp[0][IU])

    VAR[0:2] = base[0] + gx*resp[0]
    VAR[IP]  = base[1] + gx*resp[1]
    phiHbyAx = base[2] + gx*resp[2]
    phiHbyAy = base[3] + gx*resp[3]
    meanDiv0 = np.mean(np.abs(base[4] + gx*resp[4]))
    pIts     = base[5] + resp[5]

    # Step control: reject and retry rather than merely slowing down
    # ---------------------------------------------------------------
    # A backoff that only shrinks the *next* dt cannot help once a step has
    # already destroyed the state, and with free slip the momentum operator is
    # near-singular at large dt, so a bad step can grow the solution by 100x at
    # once. Roll the step back instead and repeat it smaller.
    resid = np.mean(np.abs(VARold[0:2]-VAR[0:2]))
    if not np.isfinite(resid) or resid > 10.0*residOld:
        VAR[...] = VARold
        phiHbyAx, phiHbyAy = phiOldX, phiOldY
        dtRelax = max(0.5*dtRelax, 1e-4)
        dt = dtRelax*min(dtCflPrev, dtMax)
        continue

    t += dt

    # Diagnostics
    # -----------
    divU    = fvc_divFlux(phiHbyAx, phiHbyAy)
    invTime = max(np.max(np.abs(VAR[IU]))/dx, np.max(np.abs(VAR[IV]))/dy)

    print(f'Iteration: {it}   t = {t:.3f}')
    print(f'------------------')
    print(f'                     Courant number of this step: {dt*invTime:.2f}')
    print(f'                               timestep used, dt: {dt:.3f}')
    print(f'    global continuity error: {np.sum(divU):.3e}   (sum of div(flux))')
    print(f'     final local continuity error: {np.mean(np.abs(divU)):.3e}')
    print(f'   initial local continuity error: {meanDiv0:.3e}')
    print(f'              bulk velocity / target: {bulkVelocity(VAR[IU]):.9f} / {uBulkTarget}')
    print(f'                     driving force gx: {gx:.6e}')
    print(f'       maximal velocity in field: {np.max(np.sqrt(VAR[IU]**2+VAR[IV]**2)):.6f}')
    print(f'       slip velocity at the body: {np.max(np.abs(VAR[0:2][:, nearBody])):.6f}')
    print(f'                 solution change: {np.mean(np.abs(VARold[0:2]-VAR[0:2])):.6e}')
    print(f'              cell Peclet / dtRelax: {invTime*min(dx,dy)**2/nu:.3f} / {dtRelax:.3f}')
    if linearSolver != 'direct':
        # uSolver is a single ReusedLU on the coupled route and a pair on the
        # segregated one; accept either.
        us = uSolver if isinstance(uSolver, list) else [uSolver]
        print(f'      linear solver: {pIts} CG its (p), '
              f'{sum(s.nIts for s in us)/max(sum(s.nSolve for s in us), 1):.1f} '
              f'BiCGStab its/solve (U), '
              f'{sum(s.nFactor for s in us)} factorisations so far')
    print(f'')

    # Pseudo-transient timestep for the next iteration
    # ----------------------------------------------
    # Three caps. The Courant one is the usual advective limit. The cell-Peclet
    # one guards central differencing: above Pe = 2 it loses diagonal dominance,
    # and the lagged free-slip cross term then has no margin left to be absorbed
    # into. The third is a backoff on the residual, which is what actually makes
    # this robust: with an *exact* free-slip condition a uniform tangential flow
    # lies in the null space of the steady momentum operator, so the operator is
    # nearly singular as dt grows, and how far dt can be pushed is not predicted
    # by any fixed formula (measured: the 45-degree channel diverges at dt = 10
    # for nx = 41 but converges for nx = 81, at lower cell Peclet).
    cellPe = invTime*min(dx, dy)**2/nu
    dtPe   = foMax*min(dx, dy)**2/nu/max(1.0, 0.5*cellPe)
    dtCfl  = dtMax if invTime == 0.0 else cflMax/invTime
    dtRelax = max(0.5*dtRelax, 1e-2) if resid > 1.2*residOld else min(1.5*dtRelax, 1.0)
    residOld = resid
    dtCflPrev = min(dtCfl, dtPe)
    dt = dtRelax*min(dtCfl, dtPe, dtMax)

# Save solution
VAR.tofile('var.fld')
phiHbyAx.tofile('phiHbyAx.fld')
phiHbyAy.tofile('phiHbyAy.fld')

In [ ]:
plt.figure()
plt.title('Volumes (green), fluxes (black & red) and pressure (colormap)')
xmin, xmax = Lx/2-1.3*R, Lx/2+1.3*R
ymin, ymax = Ly/2-1.3*R, Ly/2+1.3*R

theta = np.linspace(0, 2*np.pi, 100)
xBody = Lx/2 + R*np.cos(theta)
yBody = Ly/2 + R*np.sin(theta)

plt.plot(xBody, yBody, 'y')
plt.pcolormesh(XP, YP, VAR[IP])
for x in xf:
    plt.plot([x, x], [yf[0], yf[-1]], 'g', lw=0.5)
for y in yf:
    plt.plot([xf[0], xf[-1]], [y, y], 'g', lw=0.5)
plt.quiver(xf, yp, phiHbyAx, np.zeros_like(phiHbyAx), pivot='mid', scale=0.5)
plt.quiver(xp, yf, np.zeros_like(phiHbyAy), phiHbyAy, color='r', pivot='mid', scale=0.5)
for ix, x in enumerate(xp):
    for iy, y in enumerate(yp):
        if xmin < x < xmax and ymin < y < ymax:
            plt.text(x, y, f'{divU[iy, ix]:.1E}', horizontalalignment='center')
plt.colorbar()
plt.xlim([xmin, xmax])
plt.ylim([ymin, ymax])

In [ ]:
plt.figure()
plt.title('Colormap: p, vectors: u')
plt.pcolormesh(XP, YP, VAR[IP])
plt.colorbar()
plt.quiver(XP, YP, VAR[IU], VAR[IV])
plt.plot(xBody, yBody, 'y')

plt.figure()
plt.title('Colormap: u')
plt.pcolormesh(XP, YP, VAR[IU])
plt.colorbar()
plt.plot(xBody, yBody, 'y')

# Slip velocity around the body. On a grid-aligned wall this is exact; on a curved
# one the cells with both arms cut are pinned by two no-penetration conditions and
# slip much less -- see the markdown cell above.
jj, ii = np.where(nearBody)
th = np.arctan2(YP[jj, ii]-Ly/2, XP[jj, ii]-Lx/2)
ut = -np.sin(th)*VAR[IU][jj, ii] + np.cos(th)*VAR[IV][jj, ii]
un = np.cos(th)*VAR[IU][jj, ii] + np.sin(th)*VAR[IV][jj, ii]
corner = ((np.roll(solid, -1, 1) | np.roll(solid, 1, 1)) &
          (np.roll(solid, -1, 0) | np.roll(solid, 1, 0)))[jj, ii]

plt.figure()
plt.title('Velocity in the first fluid cell around the body')
plt.plot(np.degrees(th[~corner]), ut[~corner], 'o', label='tangential, one axis cut')
plt.plot(np.degrees(th[corner]),  ut[corner],  's', label='tangential, both axes cut')
plt.plot(np.degrees(th), un, '.', color='0.6', label='normal')
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('angle around the body [deg]')
plt.ylabel('velocity')
plt.legend()

plt.figure()
plt.title('Colormap: div(phi)')
plt.pcolormesh(XP, YP, divU)
plt.colorbar()